# 00 — Project and Data Inventory

## Purpose

This notebook inventories every dataset supplied under `data/raw/` without modifying any source file. It records file identity, checksums, source classification, variables, units, temporal and geographic coverage, and merge eligibility.

The authoritative sources are the original GASTAT workbooks and newly supplied Saudi-energy CSVs. Files under `technical_work_reference/` are catalogued only as derived historical artifacts and are explicitly excluded as analytical inputs.

## Outputs

- `results/data_inventory.csv`
- `results/data_dictionary.xlsx`
- `results/source_registry.csv`


## 1. Imports, Paths, and Canonical Metadata


In [ ]:
from datetime import datetime
from pathlib import Path
import hashlib
import re

import numpy as np
import openpyxl
import pandas as pd

RUN_STARTED = datetime.now().astimezone()
NOTEBOOK_DIR = Path.cwd().resolve()
if NOTEBOOK_DIR.name.lower() != "notebooks":
    raise RuntimeError(
        "Run this notebook from Saudi-Energy-Forecasting/notebooks/. "
        f"Current directory: {NOTEBOOK_DIR}"
    )

PROJECT_ROOT = NOTEBOOK_DIR.parent
RAW_DIR = PROJECT_ROOT / "data" / "raw"
GASTAT_DIR = RAW_DIR / "old_thesis_data"
SAUDI_ENERGY_DIR = RAW_DIR / "saudi_energy"
REFERENCE_DIR = RAW_DIR / "technical_work_reference"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = PROJECT_ROOT / "figures"

for directory in (INTERIM_DIR, PROCESSED_DIR, RESULTS_DIR, FIGURES_DIR):
    directory.mkdir(parents=True, exist_ok=True)

HOUSEHOLD_CSV_NAME = (
    "houses-consumption-and-cost-of-electricity-in-the-administrative-regions- (2).csv"
)
HOUSEHOLD_CSV_PATH = SAUDI_ENERGY_DIR / HOUSEHOLD_CSV_NAME

GASTAT_TABLES = {
    2019: {
        "file": "HouseholdEnergyStatistics2019En.xlsx",
        "sheet": "76",
        "table": "Table 37",
        "data_rows": (8, 20),
    },
    2020: {
        "file": "HouseholdEnergyStatistics2020En.xlsx",
        "sheet": "25",
        "table": "Houses Consumption and Cost",
        "data_rows": (8, 20),
    },
    2021: {
        "file": "HouseholdEnergyStatistics2021En.xlsx",
        "sheet": "27",
        "table": "Houses Consumption and Cost",
        "data_rows": (8, 20),
    },
    2022: {
        "file": "HouseholdEnergyStatistics2022En.xlsx",
        "sheet": "1-2",
        "table": "Table 1-2",
        "data_rows": (8, 20),
    },
}

CANONICAL_REGIONS = [
    "Riyadh",
    "Makkah",
    "Madinah",
    "Al-Qassim",
    "Eastern Region",
    "Asir",
    "Tabuk",
    "Hail",
    "Northern Borders",
    "Jazan",
    "Najran",
    "Al-Bahah",
    "Al-Jouf",
]

REGION_MAP = {
    "Riyadh": "Riyadh",
    "Makkah": "Makkah",
    "Madinah": "Madinah",
    "Qassim": "Al-Qassim",
    "Al Qassim": "Al-Qassim",
    "Al-Qassim": "Al-Qassim",
    "Eastern Region": "Eastern Region",
    "Eastern": "Eastern Region",
    "Aseer": "Asir",
    "Asir": "Asir",
    "Tabuk": "Tabuk",
    "Hail": "Hail",
    "Northern Border": "Northern Borders",
    "Northern Borders": "Northern Borders",
    "Jazan": "Jazan",
    "Najran": "Najran",
    "Al-Baha": "Al-Bahah",
    "Al Bahah": "Al-Bahah",
    "Al-Bahah": "Al-Bahah",
    "Al Jouf": "Al-Jouf",
    "Al-Jawf": "Al-Jouf",
    "Al-Jouf": "Al-Jouf",
}


def normalize_region(value):
    """Return a canonical administrative-region label without guessing unknown names."""
    if pd.isna(value):
        return pd.NA
    cleaned = re.sub(r"\s+", " ", str(value).strip().replace("–", "-").replace("—", "-"))
    return REGION_MAP.get(cleaned, cleaned)


def read_csv_flexible(path):
    """Read a supplied CSV while detecting comma/semicolon delimiters."""
    for encoding in ("utf-8-sig", "utf-8", "cp1256", "latin-1"):
        try:
            return pd.read_csv(path, sep=None, engine="python", encoding=encoding)
        except UnicodeDecodeError:
            continue
    raise UnicodeError(f"Unable to decode {path}")


def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def extract_gastat_household(year):
    """Extract one original GASTAT regional household table into canonical long form."""
    setting = GASTAT_TABLES[year]
    path = GASTAT_DIR / setting["file"]
    if not path.is_file():
        raise FileNotFoundError(path)
    workbook = openpyxl.load_workbook(path, read_only=True, data_only=True)
    if setting["sheet"] not in workbook.sheetnames:
        raise KeyError(f"Missing sheet {setting['sheet']} in {path.name}")
    sheet = workbook[setting["sheet"]]
    start_row, end_row = setting["data_rows"]
    records = []
    for excel_row, row in enumerate(
        sheet.iter_rows(min_row=start_row, max_row=end_row, values_only=True),
        start=start_row,
    ):
        values = list(row)
        if len(values) < 6 or values[1] is None:
            continue
        original_region = str(values[1]).strip()
        region = normalize_region(original_region)
        measures = [
            ("Winter", "Consumption", "kWh", values[2]),
            ("Winter", "Cost", "SAR", values[3]),
            ("Rest of the year", "Consumption", "kWh", values[4]),
            ("Rest of the year", "Cost", "SAR", values[5]),
        ]
        for season, measure, unit, raw_value in measures:
            records.append(
                {
                    "year": year,
                    "region_original": original_region,
                    "region": region,
                    "season": season,
                    "measure": measure,
                    "unit": unit,
                    "value": pd.to_numeric(raw_value, errors="coerce"),
                    "source_type": "GASTAT original workbook",
                    "source_file": setting["file"],
                    "source_sheet": setting["sheet"],
                    "source_table": setting["table"],
                    "source_excel_row": excel_row,
                }
            )
    result = pd.DataFrame(records)
    expected = len(CANONICAL_REGIONS) * 2 * 2
    if len(result) != expected:
        raise ValueError(f"{year} GASTAT extraction produced {len(result)} rows; expected {expected}")
    if set(result["region"]) != set(CANONICAL_REGIONS):
        raise ValueError(f"{year} GASTAT region coverage is not canonical")
    return result.sort_values(["year", "region", "season", "measure"]).reset_index(drop=True)


def read_household_csv():
    """Read the supplied 2017–2022 long household CSV independently."""
    if not HOUSEHOLD_CSV_PATH.is_file():
        raise FileNotFoundError(HOUSEHOLD_CSV_PATH)
    frame = read_csv_flexible(HOUSEHOLD_CSV_PATH)
    expected_columns = {
        "Year", "Administrative Region", "Season", "Measure and Unit", "Value"
    }
    if not expected_columns.issubset(frame.columns):
        raise ValueError(
            f"Household CSV columns changed. Found: {frame.columns.tolist()}"
        )
    frame = frame.rename(
        columns={
            "Year": "year",
            "Administrative Region": "region_original",
            "Season": "season",
            "Value": "value",
        }
    )
    frame["year"] = pd.to_numeric(frame["year"], errors="coerce").astype("Int64")
    frame["region"] = frame["region_original"].map(normalize_region)
    parsed = frame["Measure and Unit"].str.extract(
        r"(?P<measure>Consumption|Cost)\s*\((?P<unit>[^)]+)\)",
        expand=True,
    )
    frame["measure"] = parsed["measure"]
    frame["unit"] = parsed["unit"].replace({"KWh": "kWh", "kWh": "kWh"})
    frame["value"] = pd.to_numeric(frame["value"], errors="coerce")
    frame["source_type"] = "Independent supplied household CSV"
    frame["source_file"] = HOUSEHOLD_CSV_NAME
    frame["source_sheet"] = pd.NA
    frame["source_table"] = pd.NA
    frame["source_excel_row"] = pd.NA
    ordered = [
        "year", "region_original", "region", "season", "measure", "unit", "value",
        "source_type", "source_file", "source_sheet", "source_table", "source_excel_row",
    ]
    return frame[ordered].sort_values(
        ["year", "region", "season", "measure"]
    ).reset_index(drop=True)


print(f"Project root: {PROJECT_ROOT}")
print(f"Run started: {RUN_STARTED.isoformat(timespec='seconds')}")


## 2. Source Registry


In [ ]:
SOURCE_METADATA = {
    "HouseholdEnergyStatistics2019En.xlsx": {
        "source": "General Authority for Statistics (GASTAT)",
        "purpose": "2019 household energy survey tables",
        "temporal_resolution": "Annual survey",
        "geographic_resolution": "Primarily administrative region; some national tables",
        "authoritative": True,
    },
    "HouseholdEnergyStatistics2020En.xlsx": {
        "source": "General Authority for Statistics (GASTAT)",
        "purpose": "2020 household energy survey tables",
        "temporal_resolution": "Annual survey",
        "geographic_resolution": "Primarily administrative region; some national tables",
        "authoritative": True,
    },
    "HouseholdEnergyStatistics2021En.xlsx": {
        "source": "General Authority for Statistics (GASTAT)",
        "purpose": "2021 household energy survey tables",
        "temporal_resolution": "Annual survey",
        "geographic_resolution": "Primarily administrative region; some national tables",
        "authoritative": True,
    },
    "HouseholdEnergyStatistics2022En.xlsx": {
        "source": "General Authority for Statistics (GASTAT)",
        "purpose": "2022 household energy survey tables",
        "temporal_resolution": "Annual survey",
        "geographic_resolution": "Primarily administrative region; some national tables",
        "authoritative": True,
    },
    "HouseholdEnergyStatistics2023En.xlsx": {
        "source": "General Authority for Statistics (GASTAT)",
        "purpose": "2023 household energy survey tables",
        "temporal_resolution": "Annual survey",
        "geographic_resolution": "Administrative region indicators and national consumption",
        "authoritative": True,
    },
    HOUSEHOLD_CSV_NAME: {
        "source": "Supplied Saudi open-data extract; publisher metadata requires confirmation",
        "purpose": "Household consumption and cost by administrative region and season",
        "temporal_resolution": "Annual with winter/rest-of-year components",
        "geographic_resolution": "Administrative region and national total",
        "authoritative": True,
    },
    "electricity-consumption-indicators-by-region.csv": {
        "source": "Supplied Saudi open-data extract; publisher metadata requires confirmation",
        "purpose": "Electricity intensity in MWh per customer",
        "temporal_resolution": "Annual",
        "geographic_resolution": "Four electricity operating regions",
        "authoritative": True,
    },
    "energy-consumption-and-number-of-consumers-by-region.csv": {
        "source": "Supplied Saudi open-data extract; publisher metadata requires confirmation",
        "purpose": "Energy sales and customer counts by provider and operating region",
        "temporal_resolution": "Annual",
        "geographic_resolution": "Four electricity operating regions",
        "authoritative": True,
    },
    "saudi-arabia-electricity-load-monthly-by-region.csv": {
        "source": "Supplied Saudi open-data extract; publisher metadata requires confirmation",
        "purpose": "Monthly minimum, average, and maximum electricity load",
        "temporal_resolution": "Monthly",
        "geographic_resolution": "Four electricity operating regions",
        "authoritative": True,
    },
    "Total Number of electricity users in KSA 2024 csv.csv": {
        "source": "Supplied Saudi open-data extract citing SEC and MARAFIQ",
        "purpose": "National electricity-user counts by provider and sector",
        "temporal_resolution": "Annual",
        "geographic_resolution": "National by provider and sector",
        "authoritative": True,
    },
    "Consumer numbers and energy sales by year and category.csv": {
        "source": "Supplied Saudi open-data extract; publisher metadata requires confirmation",
        "purpose": "2023 operating-region electricity consumption",
        "temporal_resolution": "Annual",
        "geographic_resolution": "Four electricity operating regions",
        "authoritative": True,
    },
}

reference_files = sorted(REFERENCE_DIR.glob("*.csv")) if REFERENCE_DIR.exists() else []
for path in reference_files:
    SOURCE_METADATA[path.name] = {
        "source": "Historical derived reference artifact; original lineage must be reproduced",
        "purpose": "Prior processed or feature-engineered reference file",
        "temporal_resolution": "Derived",
        "geographic_resolution": "Derived",
        "authoritative": False,
    }
print(f"Registered metadata for {len(SOURCE_METADATA)} files.")


## 3. Inventory Raw Files and Workbook Sheets


In [ ]:
inventory_rows = []
sheet_rows = []
variable_rows = []

raw_files = sorted(
    path for path in RAW_DIR.rglob("*")
    if path.is_file() and path.name != ".DS_Store" and not path.name.startswith("._")
)
if not raw_files:
    raise FileNotFoundError(f"No raw files found under {RAW_DIR}")

for path in raw_files:
    metadata = SOURCE_METADATA.get(
        path.name,
        {
            "source": "Unregistered source",
            "purpose": "Requires documentation",
            "temporal_resolution": "Unknown",
            "geographic_resolution": "Unknown",
            "authoritative": False,
        },
    )
    relative_path = path.relative_to(PROJECT_ROOT).as_posix()
    base = {
        "dataset_id": path.stem.lower().replace(" ", "_"),
        "file_name": path.name,
        "relative_path": relative_path,
        "file_type": path.suffix.lower().lstrip("."),
        "size_bytes": path.stat().st_size,
        "sha256": sha256_file(path),
        **metadata,
    }

    if path.suffix.lower() == ".csv":
        frame = read_csv_flexible(path)
        numeric_years = []
        for candidate in ("Year", "year", "السنة"):
            if candidate in frame.columns:
                numeric_years = pd.to_numeric(frame[candidate], errors="coerce").dropna().astype(int).tolist()
                break
        inventory_rows.append(
            {
                **base,
                "observations_or_sheets": len(frame),
                "variable_count": frame.shape[1],
                "year_min": min(numeric_years) if numeric_years else pd.NA,
                "year_max": max(numeric_years) if numeric_years else pd.NA,
                "missing_cells": int(frame.isna().sum().sum()),
                "duplicate_rows": int(frame.duplicated().sum()),
            }
        )
        for column in frame.columns:
            variable_rows.append(
                {
                    "file_name": path.name,
                    "table_or_sheet": "CSV",
                    "variable_original": str(column),
                    "dtype_observed": str(frame[column].dtype),
                    "non_missing": int(frame[column].notna().sum()),
                    "missing": int(frame[column].isna().sum()),
                    "unique_values": int(frame[column].nunique(dropna=True)),
                    "unit": "Embedded in column/category or requires source confirmation",
                    "description": "Raw supplied variable; definition must be confirmed from source metadata",
                    "used_in_phase1_household_panel": path.name == HOUSEHOLD_CSV_NAME,
                }
            )

    elif path.suffix.lower() == ".xlsx":
        workbook = openpyxl.load_workbook(path, read_only=True, data_only=True)
        year_match = re.search(r"(20\d{2})", path.name)
        workbook_year = int(year_match.group(1)) if year_match else pd.NA
        inventory_rows.append(
            {
                **base,
                "observations_or_sheets": len(workbook.sheetnames),
                "variable_count": pd.NA,
                "year_min": workbook_year,
                "year_max": workbook_year,
                "missing_cells": pd.NA,
                "duplicate_rows": pd.NA,
            }
        )
        for sheet_name in workbook.sheetnames:
            sheet = workbook[sheet_name]
            title_text = ""
            nonempty_rows = 0
            max_nonempty_columns = 0
            for row_number, row in enumerate(
                sheet.iter_rows(min_row=1, max_row=min(sheet.max_row, 80), values_only=True),
                start=1,
            ):
                values = [value for value in row if value not in (None, "")]
                if values:
                    nonempty_rows += 1
                    max_nonempty_columns = max(max_nonempty_columns, len(values))
                    if not title_text:
                        title_text = " | ".join(str(value).replace("\n", " ").strip() for value in values)[:500]
            sheet_rows.append(
                {
                    "file_name": path.name,
                    "sheet_name": sheet_name,
                    "declared_max_rows": sheet.max_row,
                    "declared_max_columns": sheet.max_column,
                    "nonempty_rows_first_80": nonempty_rows,
                    "max_nonempty_columns_first_80": max_nonempty_columns,
                    "first_nonempty_text": title_text,
                }
            )

data_inventory = pd.DataFrame(inventory_rows).sort_values(
    ["authoritative", "relative_path"], ascending=[False, True]
).reset_index(drop=True)
excel_sheet_inventory = pd.DataFrame(sheet_rows)
variable_dictionary = pd.DataFrame(variable_rows)

# Add the canonical GASTAT household variables explicitly because they are held in multirow tables.
for year, setting in GASTAT_TABLES.items():
    for variable, unit, description in [
        ("Administrative region", "category", "Administrative region reported by GASTAT"),
        ("Winter consumption", "kWh", "Residential electricity consumption during winter"),
        ("Winter cost/value", "SAR", "Residential electricity expenditure during winter"),
        ("Rest-of-year consumption", "kWh", "Residential electricity consumption during the rest of the year"),
        ("Rest-of-year cost/value", "SAR", "Residential electricity expenditure during the rest of the year"),
    ]:
        variable_dictionary.loc[len(variable_dictionary)] = {
            "file_name": setting["file"],
            "table_or_sheet": setting["sheet"],
            "variable_original": variable,
            "dtype_observed": "numeric" if unit in {"kWh", "SAR"} else "text",
            "non_missing": 13,
            "missing": 0,
            "unique_values": 13 if unit == "category" else pd.NA,
            "unit": unit,
            "description": description,
            "used_in_phase1_household_panel": True,
        }

display(data_inventory)
print(f"Excel sheets inventoried: {len(excel_sheet_inventory):,}")
print(f"Dictionary rows: {len(variable_dictionary):,}")


## 4. Source Registry and Merge Eligibility


In [ ]:
source_registry = data_inventory[
    [
        "dataset_id", "file_name", "relative_path", "source", "purpose",
        "temporal_resolution", "geographic_resolution", "authoritative", "sha256",
    ]
].copy()

def merge_guidance(row):
    name = row["file_name"]
    if name == HOUSEHOLD_CSV_NAME:
        return "Eligible for administrative-region/year overlap comparison with GASTAT 2019–2022"
    if name.startswith("HouseholdEnergyStatistics"):
        return "Original workbook; selected household tables eligible for source-specific extraction"
    if not row["authoritative"]:
        return "Reference only; prohibited as a primary input"
    if "operating region" in str(row["geographic_resolution"]).lower():
        return "Keep separate from administrative-region household panel unless an authoritative crosswalk is supplied"
    return "Keep separate pending definition and unit validation"

source_registry["phase1_merge_guidance"] = source_registry.apply(merge_guidance, axis=1)
display(source_registry)


## 5. Save Inventory and Publication-Quality Data Dictionary


In [ ]:
inventory_path = RESULTS_DIR / "data_inventory.csv"
registry_path = RESULTS_DIR / "source_registry.csv"
dictionary_path = RESULTS_DIR / "data_dictionary.xlsx"

data_inventory.to_csv(inventory_path, index=False)
source_registry.to_csv(registry_path, index=False)

region_dictionary = pd.DataFrame(
    [{"original_label": key, "canonical_label": value} for key, value in REGION_MAP.items()]
).sort_values(["canonical_label", "original_label"])

with pd.ExcelWriter(dictionary_path, engine="openpyxl") as writer:
    data_inventory.to_excel(writer, sheet_name="Dataset Inventory", index=False)
    variable_dictionary.to_excel(writer, sheet_name="Variable Dictionary", index=False)
    excel_sheet_inventory.to_excel(writer, sheet_name="Excel Sheet Inventory", index=False)
    source_registry.to_excel(writer, sheet_name="Source Registry", index=False)
    region_dictionary.to_excel(writer, sheet_name="Region Mapping", index=False)

workbook = openpyxl.load_workbook(dictionary_path)
header_fill = openpyxl.styles.PatternFill("solid", fgColor="1F4E78")
header_font = openpyxl.styles.Font(color="FFFFFF", bold=True)
title_font = openpyxl.styles.Font(name="Calibri", size=11)
thin_gray = openpyxl.styles.Side(style="thin", color="D9E2F3")

for sheet in workbook.worksheets:
    sheet.freeze_panes = "A2"
    sheet.sheet_view.showGridLines = False
    sheet.auto_filter.ref = sheet.dimensions
    for cell in sheet[1]:
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = openpyxl.styles.Alignment(horizontal="center", vertical="center", wrap_text=True)
    for row in sheet.iter_rows(min_row=2):
        for cell in row:
            cell.font = title_font
            cell.alignment = openpyxl.styles.Alignment(vertical="top", wrap_text=True)
            cell.border = openpyxl.styles.Border(bottom=thin_gray)
    for column_cells in sheet.columns:
        letter = column_cells[0].column_letter
        max_length = max(len(str(cell.value)) if cell.value is not None else 0 for cell in column_cells[:200])
        sheet.column_dimensions[letter].width = min(max(max_length + 2, 12), 55)
    sheet.row_dimensions[1].height = 34
workbook.save(dictionary_path)

for path in (inventory_path, registry_path, dictionary_path):
    if not path.is_file() or path.stat().st_size == 0:
        raise IOError(f"Output was not saved correctly: {path}")

print("Notebook 00 complete")
print(f"- {inventory_path.relative_to(PROJECT_ROOT)}")
print(f"- {dictionary_path.relative_to(PROJECT_ROOT)}")
print(f"- {registry_path.relative_to(PROJECT_ROOT)}")


## Inventory conclusion

This inventory does not assert that differently defined geographic or sector datasets are mergeable. The administrative-region household sources are eligible for direct overlap testing. Four-region electricity-system datasets and national datasets remain separate until their definitions and any required geographic crosswalks are independently validated.
